# Non-Stationarity Analysis of Benchmark Time Series Datasets  
### Focus on ETT (ETTh1, ETTh2, ETTm1, ETTm2) for TimeBridge


## 0. Goal of this Notebook

This notebook is part of the **TimeBridge / non-stationary LTSF** project.

Here we:

1. Load the benchmark datasets used in TimeBridge (ETT, Electricity, Traffic, Weather, Solar, PeMS).
2. Diagnose **non-stationarity** using:
   - ADF (Augmented Dickey–Fuller)
   - KPSS (level & trend)
   - First differencing
   - Engle–Granger cointegration (EG)
3. Produce **heatmaps** of p-values across datasets.
4. Provide a dedicated **ETT-only section** similar to the summary in the TimeBridge paper.


#### 0.1 Setup & Helpers

In [1]:
# === 0.1 Imports, project root detection, global config ===
from pathlib import Path
import os, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss, coint

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

def find_project_root(max_up: int = 8) -> Path:
    p = Path.cwd()
    for _ in range(max_up):
        if (p / "dataset").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_ROOT    = PROJECT_ROOT / "dataset"
EXPORT_DIR   = PROJECT_ROOT / "exports" / "nonstationarity"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT   :", DATA_ROOT)
print("EXPORT_DIR  :", EXPORT_DIR)

def _coerce_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _safe_window(win, n):
    if n <= 10:
        return 2
    return int(min(max(2, win), max(2, n // 5)))

def _aligned_pair(s1, s2):
    pair = pd.concat([s1, s2], axis=1).dropna()
    return pair.iloc[:, 0].values, pair.iloc[:, 1].values

# ---------- dataset loader ----------
def load_dataset(name):
    nm = name.lower()

    # ETT
    if nm in {"etth1", "etth2", "ettm1", "ettm2"}:
        name_map = {"etth1": "ETTh1", "etth2": "ETTh2",
                    "ettm1": "ETTm1", "ettm2": "ETTm2"}
        path = DATA_ROOT / "ETT-small" / f"{name_map[nm]}.csv"
        if not path.exists():
            raise FileNotFoundError(f"ETT file not found: {path}")
        df = pd.read_csv(path)
        time_col = "date"
        value_cols = [c for c in df.columns if c != time_col]
        return _coerce_numeric(df, value_cols), time_col, value_cols

    # Electricity
    if nm == "electricity":
        path = DATA_ROOT / "electricity" / "electricity.csv"
        df = pd.read_csv(path)
        time_col = df.columns[0]
        value_cols = [c for c in df.columns if c != time_col]
        return _coerce_numeric(df, value_cols), time_col, value_cols

    # Traffic
    if nm == "traffic":
        path = DATA_ROOT / "traffic" / "traffic.csv"
        df = pd.read_csv(path)
        time_col = df.columns[0]
        value_cols = [c for c in df.columns if c != time_col]
        return _coerce_numeric(df, value_cols), time_col, value_cols

    # Weather
    if nm == "weather":
        path = DATA_ROOT / "weather" / "weather.csv"
        df = pd.read_csv(path)
        time_col = df.columns[0]
        value_cols = [c for c in df.columns if c != time_col]
        return _coerce_numeric(df, value_cols), time_col, value_cols

    # Solar
    if nm in {"solar", "solar-energy", "solar_energy"}:
        candidates = [
            DATA_ROOT / "Solar" / "solar_AL.txt",
            DATA_ROOT / "Solar" / "solar.csv",
        ]
        for p in candidates:
            if p.exists():
                try:
                    df = pd.read_csv(p)
                except Exception:
                    df = pd.read_csv(p, sep=None, engine="python")
                time_col = df.columns[0]
                value_cols = [c for c in df.columns if c != time_col]
                return _coerce_numeric(df, value_cols), time_col, value_cols
        raise FileNotFoundError("Solar file not found in expected paths.")

    # Exchange rate
    if nm in {"exchange_rate", "exchange-rate"}:
        path = DATA_ROOT / "exchange_rate" / "exchange_rate.csv"
        df = pd.read_csv(path)
        time_col = df.columns[0]
        value_cols = [c for c in df.columns if c != time_col]
        return _coerce_numeric(df, value_cols), time_col, value_cols

    # PeMS
    if nm.startswith("pems"):
        csv_path = DATA_ROOT / "PeMS" / f"{name}.csv"
        npz_path = DATA_ROOT / "PeMS" / f"{name.upper()}.npz"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            time_col = df.columns[0]
            value_cols = [c for c in df.columns if c != time_col]
            return _coerce_numeric(df, value_cols), time_col, value_cols
        if npz_path.exists():
            data = np.load(npz_path, allow_pickle=True)
            arr = data.get("data")
            if arr is None:
                for k in data.files:
                    if data[k].ndim >= 2:
                        arr = data[k]; break
            if arr is None:
                raise ValueError(f"No 2D array found inside {npz_path}")
            if "timestamps" in data:
                t = pd.to_datetime(data["timestamps"])
            else:
                t = pd.date_range("2000-01-01", periods=arr.shape[0], freq="5min")
            cols = [f"v{i}" for i in range(arr.shape[1])]
            df = pd.DataFrame(arr, columns=cols)
            df.insert(0, "time", t)
            time_col = "time"
            value_cols = cols
            return _coerce_numeric(df, value_cols), time_col, value_cols
        raise FileNotFoundError(f"PeMS file not found: {csv_path} or {npz_path}")

    raise ValueError(f"Unknown dataset name or missing file: {name}")




PROJECT_ROOT: c:\Users\badre\OneDrive\Documents\TimeBridge-Enhanced-NonStationnary-MTS
DATA_ROOT   : c:\Users\badre\OneDrive\Documents\TimeBridge-Enhanced-NonStationnary-MTS\dataset
EXPORT_DIR  : c:\Users\badre\OneDrive\Documents\TimeBridge-Enhanced-NonStationnary-MTS\exports\nonstationarity


#### 0.2 : tests de la non stationnarity 

In [2]:
def adf_test(x):
    x = pd.Series(x).dropna()
    if len(x) < 10:
        return np.nan
    _, pvalue, *_ = adfuller(x, autolag="AIC")
    return pvalue

def kpss_test(x, regression="c"):
    x = pd.Series(x).dropna()
    if len(x) < 10:
        return np.nan
    try:
        _, pvalue, *_ = kpss(x, regression=regression, nlags="auto")
    except Exception:
        pvalue = np.nan
    return pvalue

def differenced(x):
    return pd.Series(x).diff().dropna()

def analyze_series(x):
    raw_adf        = adf_test(x)
    raw_kpss_level = kpss_test(x, regression="c")
    raw_kpss_trend = kpss_test(x, regression="ct")
    dx             = differenced(x)
    d_adf          = adf_test(dx)
    d_kpss_level   = kpss_test(dx, regression="c")
    d_kpss_trend   = kpss_test(dx, regression="ct")
    return {
        "ADF p (raw)": raw_adf,
        "KPSS level p (raw)": raw_kpss_level,
        "KPSS trend p (raw)": raw_kpss_trend,
        "ADF p (diff1)": d_adf,
        "KPSS level p (diff1)": d_kpss_level,
        "KPSS trend p (diff1)": d_kpss_trend,
    }

def cointegration_scan(df, cols, max_pairs=20):
    results = []
    pairs = list(itertools.combinations(cols[:min(len(cols), 50)], 2))
    for i, j in pairs[:max_pairs]:
        x, y = _aligned_pair(df[i], df[j])
        if len(x) < 10:
            pvalue = np.nan
        else:
            try:
                pvalue = coint(x, y, trend="c")[1]
            except Exception:
                pvalue = np.nan
        results.append({"var1": i, "var2": j, "EG p": pvalue})
    return pd.DataFrame(results)

def rolling_plots(df, time_col, var, win=168, title_prefix=""):
    t = pd.to_datetime(df[time_col], errors="coerce")
    x = pd.to_numeric(df[var], errors="coerce")
    w = _safe_window(win, len(x.dropna()))
    roll_mean = x.rolling(w).mean()
    roll_std  = x.rolling(w).std()

    plt.figure()
    plt.plot(t, x, label="raw")
    plt.plot(t, roll_mean, label=f"rolling mean (win={w})")
    plt.title(f"{title_prefix}{var} — raw & rolling mean")
    plt.xlabel("time"); plt.ylabel(var); plt.legend(); plt.tight_layout(); plt.show()

    plt.figure()
    plt.plot(t, roll_std)
    plt.title(f"{title_prefix}{var} — rolling std (win={w})")
    plt.xlabel("time"); plt.ylabel("std"); plt.tight_layout(); plt.show()

def run_dataset(name, variables=3, win=168, do_coint=True, coint_pairs=10):
    df, tcol, vcols = load_dataset(name)
    vcols = [c for c in vcols if pd.api.types.is_numeric_dtype(df[c])]
    if not vcols:
        raise ValueError(f"No numeric value columns found in {name}")
    take = vcols[:min(variables, len(vcols))]

    rows = []
    for col in take:
        rolling_plots(df, tcol, col, win=win, title_prefix=f"[{name}] ")
        stats = analyze_series(df[col].values)
        stats.update({"dataset": name, "variable": col})
        rows.append(stats)

    res = pd.DataFrame(rows)
    out_csv = EXPORT_DIR / f"{name}_stationarity_summary.csv"
    res.to_csv(out_csv, index=False)
    print(f"Summary saved: {out_csv}")

    if do_coint and len(vcols) >= 2:
        eg = cointegration_scan(df, vcols, max_pairs=coint_pairs)
        eg_csv = EXPORT_DIR / f"{name}_cointegration_pairs.csv"
        eg.to_csv(eg_csv, index=False)
        print(f"Cointegration pairs saved: {eg_csv}")

    return res

In [3]:
from itertools import combinations

ett_list = [
    ("ETTm1", "15 min"),
    ("ETTm2", "15 min"),
    ("ETTh1", "1 hour"),
    ("ETTh2", "1 hour"),
]

def adf_logsum(series_list):
    pvals = []
    for s in series_list:
        x = pd.Series(s).dropna()
        if len(x) < 10:
            pvals.append(np.nan); continue
        _, p, *_ = adfuller(x, autolag="AIC")
        p = max(p, 1e-300)
        pvals.append(p)
    return np.nansum(np.log10(np.array(pvals, dtype=float)))

def eg_count(df, cols, alpha=0.05):
    cnt = 0
    for i, j in combinations(cols, 2):
        pair = pd.concat([df[i], df[j]], axis=1).dropna()
        if len(pair) < 10:
            continue
        try:
            pv = coint(pair.iloc[:, 0].values, pair.iloc[:, 1].values, trend="c")[1]
            if pv < alpha:
                cnt += 1
        except Exception:
            pass
    return cnt

rows = []
for name, freq in ett_list:
    df, tcol, vcols = load_dataset(name)
    vcols = vcols[:7]
    dim = len(vcols)

    adf_score = adf_logsum([df[c].values for c in vcols])
    eg_pairs  = eg_count(df, vcols, alpha=0.05)

    rows.append({
        "Dataset": name,
        "Dim": dim,
        "Frequency": freq,
        "ADF_score (Σ log10 p)": adf_score,
        "EG_count (p<0.05)": eg_pairs,
    })

summary_ett = pd.DataFrame(rows).set_index("Dataset")
summary_ett


,Dim,Frequency,ADF_score (Σ log10 p),EG_count (p<0.05)
Dataset,,,,
ETTm1,7,15 min,-682.246827,21
ETTm2,7,15 min,-45.604623,20
ETTh1,7,1 hour,-48.442871,21
ETTh2,7,1 hour,-23.340311,18


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# collect per-variable p-values for the 4 ETT datasets
records = []
for name, _ in ett_list:   # ett_list = [("ETTm1","15 min"), ...]
    df, tcol, vcols = load_dataset(name)
    vcols = vcols[:7]      # 7 variables
    for var in vcols:
        stats = analyze_series(df[var].values)
        stats["dataset"] = name
        stats["variable"] = var
        records.append(stats)

ett_pvals = pd.DataFrame(records)

cols = [
    "ADF p (raw)", "KPSS level p (raw)", "KPSS trend p (raw)",
    "ADF p (diff1)", "KPSS level p (diff1)", "KPSS trend p (diff1)",
]

# mean p-value over the 7 variables
heat = (
    ett_pvals.groupby("dataset")[cols]
             .mean()
             .reindex([n for n, _ in ett_list])
)

plt.figure(figsize=(10, 4))
sns.heatmap(
    heat,
    annot=True,
    fmt=".2e",                 # scientific notation, e.g. 3.4e-06
    cmap="coolwarm_r",
    vmin=0.0, vmax=1.0,
    cbar_kws={"label": "p-value"}
)
plt.title("ETT: Average ADF/KPSS p-values across 7 variables (raw scale)")
plt.ylabel("Dataset")
plt.xlabel("Test")
plt.tight_layout()
plt.show()
